# Variational Quantum Classifier (VQC) for CICIDS2017 (Full Scale)

This notebook implements a **Hybrid Quantum-Classical Classifier** for Network Intrusion Detection using PennyLane on the **full CICIDS2017 dataset**.

**Model Architecture:**
- Dataset: CICIDS2017 (Full Scale, all files concatenated)
- Preprocessing: PCA (Dimensionality reduction → 8 features)
- Quantum Circuit: 8-qubit variational circuit with Strongly Entangling Layers
- Framework: PennyLane + PyTorch

## 1. Setup and Imports

In [ ]:
import sys
import os
import glob
from pathlib import Path
import pickle
from tqdm.auto import tqdm

# Add project root to path
project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# PennyLane
import pennylane as qml
from pennylane import numpy as pnp

# Custom Model Import
from src.models.quantum.pennylane_models import HybridQuantumClassifier

sns.set_style('whitegrid')
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"Setup Complete. PennyLane: {qml.__version__}")

## 2. Full Scale Data Loading (CICIDS2017)

We will load and concatenate ALL CSV files from the CICIDS2017 dataset to ensure full representation of the network traffic.

In [ ]:
DATA_DIR = project_root / 'data' / 'raw' / 'cicids2017'
csv_files = glob.glob(str(DATA_DIR / "*.csv"))
print(f"Found {len(csv_files)} files.")

def load_full_cicids17(files):
    all_chunks = []
    for file in tqdm(files, desc="Loading files"):
        try:
            # Read the full file
            df = pd.read_csv(file)
            
            # Clean column names (strip spaces)
            df.columns = [str(c).strip() for c in df.columns]
            
            # Process labels: 'BENIGN' -> 0, everything else -> 1
            if 'Label' in df.columns:
                df['binary_label'] = df['Label'].apply(lambda x: 0 if str(x).upper() == 'BENIGN' else 1)
                df = df.drop('Label', axis=1)
            
            # Drop metadata if exists (though usually not in ISCX ML versions)
            drop_cols = ['Flow ID', 'Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Timestamp']
            df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')
            
            # Handle non-numeric artifacts
            df = df.apply(pd.to_numeric, errors='coerce')
            
            # Drop infinities and nans
            df = df.replace([np.inf, -np.inf], np.nan).dropna()
            
            all_chunks.append(df)
        except Exception as e:
            print(f"Error reading {file}: {e}")
            
    return pd.concat(all_chunks, ignore_index=True)

print("Building full dataset (No sampling caps)...")
full_df = load_full_cicids17(csv_files)
print(f"Final Dataset Size: {full_df.shape}")
print("Class Distribution:")
print(full_df['binary_label'].value_counts())
print(f"Attack Ratio: {full_df['binary_label'].mean():.2%}")

## 3. Preprocessing

Dimensionality reduction is critical for quantum simulation. We will use PCA to reduce the feature space to 8 features, matching our quantum circuit's capacity.

In [ ]:
X = full_df.drop('binary_label', axis=1)
y = full_df['binary_label'].values

print(f"Splitting dataset (stratified)... ")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

N_QUBITS = 8
pca = PCA(n_components=N_QUBITS)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(f"Original features: {X_train.shape[1]} -> PCA reduced to: {N_QUBITS}")
print(f"Explained variance: {np.sum(pca.explained_variance_ratio_):.4f}")

## 4. Model & DataLoaders

In [ ]:
train_ds = TensorDataset(torch.FloatTensor(X_train_pca), torch.LongTensor(y_train))
test_ds = TensorDataset(torch.FloatTensor(X_test_pca), torch.LongTensor(y_test))

# Using a slightly larger batch size for full-scale data efficiency
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=128)

model = HybridQuantumClassifier(
    input_dim=N_QUBITS,
    num_classes=2,
    n_qubits=N_QUBITS,
    n_quantum_layers=2
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
print(f"Training on {device}")
print(model)

## 5. Training Loop

The training will execute on the full dataset without any sampling caps.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)
epochs = 10

print("Starting Quantum Hybrid Training (Full Scale)... ")
epoch_pbar = tqdm(range(epochs), desc="Training Progress")

for epoch in epoch_pbar:
    model.train()
    total_loss = 0
    correct = 0
    
    batch_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False)
    for X_batch, y_batch in batch_pbar:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        preds = torch.argmax(outputs, dim=1)
        item_correct = (preds == y_batch).sum().item()
        correct += item_correct
        
        batch_pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{item_correct/len(y_batch):.2f}'})
        
    avg_loss = total_loss / len(train_loader)
    acc = correct / len(train_ds)
    epoch_pbar.set_postfix({'avg_loss': f'{avg_loss:.4f}', 'total_acc': f'{acc:.4f}'})

## 6. Evaluation

In [ ]:
model.eval()
all_preds = []
with torch.no_grad():
    for X_batch, y_batch in tqdm(test_loader, desc="Evaluating Full Set"):
        X_batch = X_batch.to(device)
        outputs = model(X_batch)
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())

print("\nClassification Report (Full Scale CICIDS2017):")
print(classification_report(y_test, all_preds, target_names=['Benign', 'Attack']))

cm = confusion_matrix(y_test, all_preds)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Full Scale CICIDS2017')
plt.show()

## 7. Saving Artifacts

In [ ]:
SAVE_DIR = project_root / 'results' / 'models' / 'quantum'
SAVE_DIR.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
model_path = SAVE_DIR / f'vqc_hybrid_cicids2017_full_{timestamp}.pt'
pre_path = SAVE_DIR / f'vqc_cicids2017_preprocessing_full_{timestamp}.pkl'

torch.save({
    'model_state_dict': model.state_dict(),
    'model_config': {
        'input_dim': N_QUBITS,
        'num_classes': 2,
        'n_qubits': N_QUBITS,
        'n_quantum_layers': 2
    }
}, model_path)

with open(pre_path, 'wb') as f:
    pickle.dump({
        'scaler': scaler,
        'pca': pca,
        'feature_cols': X.columns.tolist()
    }, f)

print(f"Final Scale Artifacts saved to {SAVE_DIR}")

In [ ]:
torch.cuda.empty_cache()
import gc
gc.collect()